<a href="https://colab.research.google.com/github/betmutema/ml-engineering-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/betmutema/ml-engineering-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm choosing Lane 2 (Refresh/Content Opportunity Scoring). I want to treat webpages as system components with a lifecycle, and use their observed performance data to flag which ones need maintenance (a refresh) before they lose most of their value. This lane fits because the starter pipeline already ships a working baseline + model comparison for exactly this kind of decline signal, which gives me something concrete to beat and improve on over the next 7 weeks.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Which pages should a content editor review first for a refresh, out of a much larger pool than they have time to check manually.

**Who acts:** The content/SEO editor with limited review capacity and they can't check every page every week.

**Action:** Update, expand or otherwise refresh the flagged page; or if the model is wrong, leave a healthy page alone or miss a page that keeps declining.

**Cost of a wrong call:** A false positive (flagging a healthy page) wastes editor hours, a labor cost. A false negative (missing a truly declining page) means it keeps losing traffic/visibility until someone notices, a lost-visibility cost that compounds the longer it's missed.

**Why ML, not just a rule:** A single threshold rule (e.g."flag anything with dropping traffic") ignores that decline shows up differently depending on position, content age and demand level, the pattern is real but too tangled across many signals to hand-write reliably which the starter pipeline's own baseline-vs-model comparison already demonstrates (baseline precision@50 = 0.240 vs random forest = 0.740).

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [6]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/betmutema/ml-engineering-internship"  # my fork
REPO_DIR = "ml-engineering-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks" and os.path.basename(os.path.dirname(os.getcwd())) == "work":
    os.chdir("../..")  # moved from work/notebooks/ up two levels to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/ml-engineering-internship
Starter data found. You're ready.


In [7]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Apply the required filters from the lane guide / starter pipeline:
# only pages old enough and visible enough to matter
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates(subset="content_id")

print(f"Rows after filtering: {len(df):,}")
print(f"Unique clients: {df['client_id'].nunique()}")

# Number 1: how much of the pool is actually declining?
declining_share = (df["trend_direction"] == "down").mean()
print(f"\nShare of pages currently declining: {declining_share:.1%}")

# Number 2: how many pages have real visibility AND real position
# (avg_position == 0 means "no data" -- exclude it, per the data skill's warning)
visible_ranked = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)]
print(f"Pages with 500+ impressions and a real ranking position: {len(visible_ranked):,} "
      f"({len(visible_ranked) / len(df):.1%} of the filtered pool)")

# Number 3: among visible pages, how many are BOTH declining and high-demand
# -- this is roughly the "declining_with_demand" reason code from the starter baseline
declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
print(f"Declining pages with real demand (impressions_90d >= 100): {len(declining_with_demand):,}")

Rows after filtering: 30,000
Unique clients: 32

Share of pages currently declining: 54.2%
Pages with 500+ impressions and a real ranking position: 16,726 (55.8% of the filtered pool)
Declining pages with real demand (impressions_90d >= 100): 13,152


Working from the starter dataset (30,000 rows, 32 clients filtering for demand and age removed nothing, since the starter CSV ships already meeting those bars), I found:

* 54.2% of pages are currently in decline (trend_direction == "down") which is more than half the pool, which is far too many for manual review at this scale.

* Of those declining pages, 13,152 also have real demand (90-day impressions ≥ 100) and so the problem isn't noise from dead pages; it's happening to pages that still matter.

* 16,726 pages (55.8%) have both meaningful traffic (≥500 impressions) and a real search ranking position, showing there's a large, well-measured pool to prioritize within.

Together these numbers show the review queue this lane would need to prioritize is large and mostly not low-stakes which is exactly the kind of "too many candidates, no obvious single rule" situation this lane guide says a learned ranking is meant to help with (backed by the starter pipeline's own baseline-vs-model comparison: precision@50 of 0.240 vs. 0.740).

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

Based on the starter dataset, over half of tracked pages (54.2%) are currently declining, and a large share of those (13,152) still have meaningful search demand which is a real, common pattern worth prioritizing and not a rare edge case. The starter pipeline's own results show a learned ranking model can identify high-priority pages far more precisely than a hand-written rule (precision@50 of 0.740 vs. 0.240) showing a ~3x improvement in how many of the top 50 flagged pages are actually worth reviewing.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.